# Validate the Azure ML Workshop

This read-only notebook validates the cloned workshop, the single `.env`, local tools, Azure authentication, workspace access, compute access, and representative predeployed assets.

## Sources

The structure follows this repository's Azure ML SDK v2 notebooks and the [Azure ML examples repository](https://github.com/Azure/azureml-examples). See `workshop/SOURCES.md` for the source and license ledger.

In [ ]:
# 1. Import Required Libraries
from itertools import islice
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

In [ ]:
# 2. Define Configuration
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (
        (candidate / ".env.example").is_file()
        and (candidate / "pipelines").is_dir()
        and (candidate / "notebooks").is_dir()
    ):
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

ENV_FILE = WORKSHOP_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError("Copy workshop/.env.example to workshop/.env and provide workspace values")
load_dotenv(ENV_FILE, override=True)

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
TENANT_ID = os.getenv("AZURE_TENANT_ID", "").strip()
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "").strip()
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "").strip()
COMPUTE_NAME = os.getenv("AZUREML_COMPUTE_NAME", "").strip()
H2O_VERSION = os.getenv("H2O_VERSION", "").strip()

required = {
    "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID,
    "AZURE_RESOURCE_GROUP": RESOURCE_GROUP,
    "AZUREML_WORKSPACE_NAME": WORKSPACE_NAME,
    "AZUREML_COMPUTE_NAME": COMPUTE_NAME,
    "H2O_VERSION": H2O_VERSION,
}
missing = [name for name, value in required.items() if not value or value.startswith("<")]
if missing:
    raise ValueError("Missing or placeholder .env values: " + ", ".join(missing))

credential = AzureCliCredential(tenant_id=TENANT_ID or None)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

In [ ]:
# 3. Implement Core Functions
def command_available(name: str) -> bool:
    """Return whether a command is available on PATH."""
    if not name.strip():
        raise ValueError("Command name cannot be empty")
    return shutil.which(name) is not None


def first_names(items, limit: int = 5) -> list[str]:
    """Return up to limit asset names without exhausting a paged Azure iterator."""
    if limit < 1:
        raise ValueError("Limit must be positive")
    return [item.name for item in islice(items, limit)]


def add_rows(rows: list[dict], asset_type: str, names: list[str]) -> None:
    """Append display rows for a named Azure ML asset collection."""
    rows.extend({"asset_type": asset_type, "name": name} for name in names)

In [ ]:
# 4. Run the Implementation
workspace = ml_client.workspaces.get(WORKSPACE_NAME)
compute = ml_client.compute.get(COMPUTE_NAME)

print(f"Workspace: {workspace.name}")
print(f"Resource group: {workspace.resource_group}")
print(f"Compute: {compute.name} ({compute.type})")
print(f"Python: {sys.version.split()[0]}")
print({name: command_available(name) for name in ("az", "git", "java", "uv")})

rows = []
add_rows(rows, "data", first_names(ml_client.data.list()))
add_rows(rows, "model", first_names(ml_client.models.list()))
add_rows(rows, "environment", first_names(ml_client.environments.list()))
add_rows(rows, "job", first_names(ml_client.jobs.list()))
add_rows(rows, "endpoint", first_names(ml_client.online_endpoints.list()))
asset_summary = pd.DataFrame(rows)
display(asset_summary)

In [ ]:
# 5. Validate Expected Behavior
assert WORKSHOP_ROOT.name == "workshop"
assert workspace.name == WORKSPACE_NAME
assert compute.name == COMPUTE_NAME
assert (WORKSHOP_ROOT / "pipelines/single-step-merge-job.yaml").is_file()
assert (WORKSHOP_ROOT / "pipelines/integration-compare-pipeline.yaml").is_file()
assert (WORKSHOP_ROOT / "pipelines/h2o-customer-scoring-pipeline.yaml").is_file()
assert command_available("az")
assert command_available("git")
assert command_available("java")
print("Workshop preflight passed. No Azure resources were changed.")

## Expected Result and Next Step

The notebook reports the configured workspace and compute, confirms required local tools, and shows representative predeployed assets without changing Azure.

Next: `notebooks/01_foundations/01_workspace_and_compute.ipynb`.